# FYP End-to-End Pipeline Demo
## Hallucination Detection & KG-Augmented Automated Program Repair

This notebook demonstrates the full pipeline:
1. **Input**: A code snippet generated by an LLM that contains an error (hallucination)
2. **Static Analysis**: AST analysis, CFG analysis, Library/API validation
3. **Dynamic Execution**: Runtime testing to catch errors
4. **Fault Aggregation**: Combining all error sources
5. **Patch Generation**: Marking error regions in the code
6. **KG Suggestions**: Querying the DS-KG Knowledge Graph for API corrections
7. **LLM Repair**: Using Qwen2.5-Coder-3B-Instruct to generate the fixed code

**Example used**: Task DS0042 from DS1000 — generated code uses `DataFrame.append()` which was removed in newer pandas versions.

In [ ]:
# ============================================================
# Install all required dependencies
# ============================================================
!pip install pandas numpy transformers torch accelerate -q

: 

In [ ]:
# ============================================================
# Imports — all standard library + third-party imports
# ============================================================
import ast
import json
import re
import os
import sys
import glob
import traceback
import importlib
import inspect
import builtins
from typing import List, Tuple, Optional
from difflib import get_close_matches
from pprint import pprint

import pandas as pd
import numpy as np

print("All imports loaded successfully.")

## Step 0 — Input: Buggy Generated Code

Task **DS0042** from the DS1000 dataset. The Qwen model generated code that uses `DataFrame.append()`, which was **removed in pandas 2.0+**. This is a classic API hallucination.

In [ ]:
# ============================================================
# Define the buggy generated code (DS0042 from DS1000)
# ============================================================

buggy_code = """\
import pandas as pd
import numpy as np
df = test_input
result = df.iloc[1:].reset_index(drop=True).add_prefix('A_')
result = result.append(df.iloc[0].rename(columns=lambda x: f'B_{x}'))
"""

# Display with line numbers
print("=" * 60)
print("BUGGY GENERATED CODE (DS0042)")
print("=" * 60)
for i, line in enumerate(buggy_code.strip().split('\n'), 1):
    print(f"  {i:>3} | {line}")
print("=" * 60)

## Step 1 — AST Analysis (Static)

Checks for **syntax errors**, **indentation errors**, and **structural violations** (e.g., `return` outside a function, `break` outside a loop).

Source: `Hallucination detection/static/AST/ast_analysis.py`

In [ ]:
# ============================================================
# AST Analysis — from ast_analysis.py
# ============================================================

class StructuralViolationVisitor(ast.NodeVisitor):
    """Detects structural violations: return outside function,
    break/continue outside loop."""

    def __init__(self):
        self.errors = []
        self.in_function = 0
        self.in_loop = 0

    def _record(self, error_type, node):
        start = getattr(node, "lineno", None)
        end = getattr(node, "end_lineno", start)
        self.errors.append({
            "type": error_type,
            "start_line": start,
            "end_line": end
        })

    def visit_FunctionDef(self, node):
        self.in_function += 1
        self.generic_visit(node)
        self.in_function -= 1

    def visit_AsyncFunctionDef(self, node):
        self.in_function += 1
        self.generic_visit(node)
        self.in_function -= 1

    def visit_For(self, node):
        self.in_loop += 1
        self.generic_visit(node)
        self.in_loop -= 1

    def visit_While(self, node):
        self.in_loop += 1
        self.generic_visit(node)
        self.in_loop -= 1

    def visit_Return(self, node):
        if self.in_function == 0:
            self._record("return_outside_function", node)
        self.generic_visit(node)

    def visit_Break(self, node):
        if self.in_loop == 0:
            self._record("break_outside_loop", node)

    def visit_Continue(self, node):
        if self.in_loop == 0:
            self._record("continue_outside_loop", node)


def analyze_ast(code: str) -> dict:
    """Analyze code for syntax errors, indentation errors,
    and structural violations using AST parsing."""
    result = {
        "ast_parsed": False,
        "syntax_error": 0,
        "indentation_error": 0,
        "structural_error": 0,
        "error_type": None,
        "line": None,
        "message": None,
        "structural_details": []
    }

    try:
        tree = ast.parse(code)
        result["ast_parsed"] = True

        visitor = StructuralViolationVisitor()
        visitor.visit(tree)

        if visitor.errors:
            result["structural_error"] = len(visitor.errors)
            result["structural_details"] = visitor.errors

    except IndentationError as e:
        result["indentation_error"] = 1
        result["error_type"] = "IndentationError"
        result["line"] = e.lineno
        result["message"] = str(e)

    except SyntaxError as e:
        result["syntax_error"] = 1
        result["error_type"] = "SyntaxError"
        result["line"] = e.lineno
        result["message"] = str(e)

    return result


# --- Run AST analysis ---
ast_result = analyze_ast(buggy_code)

print("=" * 60)
print("AST ANALYSIS RESULT")
print("=" * 60)
pprint(ast_result)
print()
if ast_result["ast_parsed"]:
    print(">> AST parsed successfully — no syntax/indentation errors.")
    if ast_result["structural_error"] > 0:
        print(f">> Found {ast_result['structural_error']} structural violation(s).")
    else:
        print(">> No structural violations found.")
else:
    print(f">> Parse FAILED: {ast_result['error_type']} at line {ast_result['line']}")
    print(f"   {ast_result['message']}")

## Step 2 — CFG Analysis (Static)

Checks for **unreachable code** (statements after `return`/`raise`/`break`/`continue`) and **missing return** paths in functions.

Source: `Hallucination detection/static/CFG/cfg_analysis.py`

In [ ]:
# ============================================================
# CFG Analysis — from cfg_analysis.py
# ============================================================

class CFGVisitor(ast.NodeVisitor):
    """Detects unreachable code and missing return paths."""

    def __init__(self):
        self.unreachable = []
        self.missing_returns = []

    def _check_block_unreachable(self, statements):
        terminated = False
        for stmt in statements:
            if terminated:
                self.unreachable.append({
                    "type": "unreachable_code",
                    "start_line": stmt.lineno,
                    "end_line": getattr(stmt, "end_lineno", stmt.lineno)
                })
            if isinstance(stmt, (ast.Return, ast.Raise, ast.Break, ast.Continue)):
                terminated = True

    def visit_FunctionDef(self, node):
        self._check_block_unreachable(node.body)
        self._check_missing_return(node)
        self.generic_visit(node)

    def visit_AsyncFunctionDef(self, node):
        self._check_block_unreachable(node.body)
        self._check_missing_return(node)
        self.generic_visit(node)

    def _block_returns(self, stmts):
        for stmt in stmts:
            if isinstance(stmt, ast.Return):
                return True
            if isinstance(stmt, ast.If):
                if stmt.orelse:
                    if self._block_returns(stmt.body) and self._block_returns(stmt.orelse):
                        return True
                else:
                    return False
        return False

    def _check_missing_return(self, node):
        if not self._block_returns(node.body):
            self.missing_returns.append({
                "type": "missing_return",
                "function": node.name,
                "start_line": node.lineno,
                "end_line": getattr(node, "end_lineno", node.lineno)
            })


def analyze_cfg(code: str) -> dict:
    """Analyze code for control flow issues: unreachable code
    and missing return paths."""
    result = {
        "cfg_analyzed": False,
        "unreachable_code": 0,
        "missing_return": 0,
        "cfg_details": []
    }

    try:
        tree = ast.parse(code)
        visitor = CFGVisitor()
        visitor.visit(tree)

        result["cfg_analyzed"] = True
        result["unreachable_code"] = len(visitor.unreachable)
        result["missing_return"] = len(visitor.missing_returns)
        result["cfg_details"] = visitor.unreachable + visitor.missing_returns

    except Exception:
        pass
    return result


# --- Run CFG analysis ---
cfg_result = analyze_cfg(buggy_code)

print("=" * 60)
print("CFG ANALYSIS RESULT")
print("=" * 60)
pprint(cfg_result)
print()
if cfg_result["cfg_analyzed"]:
    print(f">> Unreachable code blocks: {cfg_result['unreachable_code']}")
    print(f">> Missing return paths:    {cfg_result['missing_return']}")
    if cfg_result["cfg_details"]:
        print(">> Details:")
        for d in cfg_result["cfg_details"]:
            pprint(d)
    else:
        print(">> No CFG issues found.")
else:
    print(">> CFG analysis could not be performed (parse failure).")

## Step 3 — Library/API Analysis (Static)

Validates **import statements**, **attribute access** on imported modules, and **function call signatures** against the actual runtime modules.

Source: `Hallucination detection/static/LIB_API/library_api.py`

In [ ]:
# ============================================================
# Library/API Analysis — from library_api.py
# ============================================================

class LibraryAPIVisitor(ast.NodeVisitor):
    """Validates imports, attribute access, and function call
    signatures against actual runtime modules."""

    def __init__(self):
        self.imports = {}
        self.errors = []

    # ---------- Imports ----------
    def visit_Import(self, node):
        for alias in node.names:
            name = alias.asname or alias.name.split(".")[0]
            try:
                self.imports[name] = importlib.import_module(alias.name)
            except Exception:
                self.errors.append({
                    "type": "module_not_found",
                    "module": alias.name,
                    "line": node.lineno
                })

    def visit_ImportFrom(self, node):
        if node.module is None:
            return
        try:
            module = importlib.import_module(node.module)
            for alias in node.names:
                name = alias.asname or alias.name
                if hasattr(module, alias.name):
                    self.imports[name] = getattr(module, alias.name)
                else:
                    self.errors.append({
                        "type": "name_error",
                        "name": alias.name,
                        "line": node.lineno
                    })
        except Exception:
            self.errors.append({
                "type": "module_not_found",
                "module": node.module,
                "line": node.lineno
            })

    # ---------- Attribute Access ----------
    def visit_Attribute(self, node):
        if isinstance(node.value, ast.Name):
            base = node.value.id
            attr = node.attr
            if base in self.imports:
                obj = self.imports[base]
                if not hasattr(obj, attr):
                    self.errors.append({
                        "type": "attribute_error",
                        "object": base,
                        "attribute": attr,
                        "line": node.lineno
                    })
        self.generic_visit(node)

    # ---------- Function Calls ----------
    def visit_Call(self, node):
        if isinstance(node.func, ast.Attribute):
            if isinstance(node.func.value, ast.Name):
                base = node.func.value.id
                func_name = node.func.attr
                if base in self.imports:
                    obj = self.imports[base]
                    if hasattr(obj, func_name):
                        try:
                            sig = inspect.signature(getattr(obj, func_name))
                            for kw in node.keywords:
                                if kw.arg not in sig.parameters:
                                    self.errors.append({
                                        "type": "type_error",
                                        "function": func_name,
                                        "invalid_arg": kw.arg,
                                        "line": node.lineno
                                    })
                        except Exception:
                            pass
        self.generic_visit(node)


def analyze_library_api(code: str) -> dict:
    """Analyze code for library/API misuse: bad imports,
    wrong attributes, invalid function arguments."""
    result = {
        "libapi_analyzed": False,
        "name_error": 0,
        "attribute_error": 0,
        "type_error": 0,
        "module_not_found": 0,
        "total_libapi_errors": 0,
        "libapi_details": []
    }

    try:
        tree = ast.parse(code)
        visitor = LibraryAPIVisitor()
        visitor.visit(tree)

        result["libapi_analyzed"] = True
        result["libapi_details"] = visitor.errors

        for err in visitor.errors:
            result[err["type"]] += 1

        result["total_libapi_errors"] = len(visitor.errors)

    except Exception:
        pass

    return result


# --- Run Library/API analysis ---
libapi_result = analyze_library_api(buggy_code)

print("=" * 60)
print("LIBRARY/API ANALYSIS RESULT")
print("=" * 60)
pprint(libapi_result)
print()
if libapi_result["libapi_analyzed"]:
    print(f">> Total Library/API errors: {libapi_result['total_libapi_errors']}")
    if libapi_result["libapi_details"]:
        print(">> Error details:")
        for d in libapi_result["libapi_details"]:
            pprint(d)
    else:
        print(">> No Library/API errors detected via static analysis.")
else:
    print(">> Library/API analysis could not be performed.")

## Step 4 — Dynamic Execution

Actually **execute** the generated code with a sample `test_input` DataFrame and capture the runtime error. This mirrors `Hallucination detection/dynamic/dynamic_execution.py`.

Source: `Hallucination detection/dynamic/dynamic_execution.py`

In [ ]:
# ============================================================
# Dynamic Execution — simplified from dynamic_execution.py
# ============================================================

# Create a sample test_input DataFrame matching DS0042's test format
test_input = pd.DataFrame({
    'Nanonose': ['Sample type', 'Water', 'Water', 'Water', 'Water'],
    'Unnamed: 1': ['Concentration', 9200, 9200, 9200, 4600],
    'A': [np.nan, 95.5, 94.5, 92.0, 53.0],
    'B': [np.nan, 21.0, 17.0, 16.0, 7.5],
    'C': [np.nan, 6.0, 5.0, 3.0, 2.5],
    'D': [np.nan, 11.942308, 5.484615, 11.057692, 3.538462],
    'E': [np.nan, 64.134615, 63.205769, 62.586538, 35.163462],
    'F': [np.nan, 21.49856, 19.65856, 19.81312, 6.876207],
    'G': [np.nan, 5.56784, 4.968, 5.19248, 1.641724],
    'H': [np.nan, 1.174135, 1.883444, 0.564835, 0.144654]
})

# Execute the buggy code and capture error info
dynamic_result = {
    "status": "passed",
    "error_type": "",
    "error_message": "",
    "line_number": "",
}

try:
    exec_env = {"test_input": test_input}
    exec(buggy_code, exec_env)
    dynamic_result["status"] = "passed"
except Exception as e:
    dynamic_result["status"] = "failed"
    dynamic_result["error_type"] = type(e).__name__
    dynamic_result["error_message"] = str(e)

    # Extract line number from traceback
    tb = traceback.extract_tb(e.__traceback__)
    string_frames = [frame for frame in tb if '<string>' in frame.filename]
    if string_frames:
        dynamic_result["line_number"] = str(string_frames[-1].lineno)


print("=" * 60)
print("DYNAMIC EXECUTION RESULT")
print("=" * 60)
pprint(dynamic_result)
print()
if dynamic_result["status"] == "failed":
    print(f">> Runtime error: {dynamic_result['error_type']}")
    print(f">> Message: {dynamic_result['error_message']}")
    print(f">> At line: {dynamic_result['line_number']}")
else:
    print(">> Code executed successfully (no runtime error).")

## Step 5 — Fault Information Aggregation

Combine all error information from AST, CFG, Library/API, and Dynamic analysis into a single **fault information** summary, mirroring the format used in `fault_information.csv`.

In [ ]:
# ============================================================
# Fault Information Aggregation
# ============================================================

# Build ast_info (matches fault_information.csv format)
ast_info = ""
if ast_result["error_type"]:
    ast_info = json.dumps({
        "type": ast_result["error_type"],
        "value": ast_result["line"],
        "message": ast_result["message"]
    })

# Build cfg_info
cfg_info = ""
if cfg_result["cfg_details"]:
    cfg_info = str(cfg_result["cfg_details"])

# Build lib_info
lib_info = ""
if libapi_result["libapi_details"]:
    lib_info = str(libapi_result["libapi_details"])

# Build dynamic_info
dynamic_info = ""
if dynamic_result["status"] == "failed":
    dynamic_info = json.dumps({
        "error_type": dynamic_result["error_type"],
        "error_message": dynamic_result["error_message"],
        "line_no": dynamic_result["line_number"],
    })

# Combined fault information dict
fault_info = {
    "dataset": "DS1000",
    "task_id": "DS0042",
    "status": "hallucinated" if dynamic_result["status"] == "failed" else "passed",
    "ast_info": ast_info,
    "cfg_info": cfg_info,
    "lib_info": lib_info,
    "dynamic_info": dynamic_info,
}

print("=" * 60)
print("FAULT INFORMATION SUMMARY")
print("=" * 60)
for key, val in fault_info.items():
    display_val = val if val else "(empty — no errors from this source)"
    print(f"  {key:>15}: {display_val}")
print()

# Count error sources
error_sources = []
if ast_info: error_sources.append("AST")
if cfg_info: error_sources.append("CFG")
if lib_info: error_sources.append("Library/API")
if dynamic_info: error_sources.append("Dynamic")

if error_sources:
    print(f">> Errors detected from: {', '.join(error_sources)}")
else:
    print(">> No errors detected from any source.")

## Step 6 — Patch Generation

Generate the **entire code file** with `<<<< [ERROR START] (source: type)` / `[ERROR FINISH] (source: type) >>>>` markers inserted at every detected error location. This matches the format produced by `patch_generator.py` and stored in `patched_code.csv` — the LLM receives the complete program so it can see the full context while knowing exactly where each error is.

Source: `patch_generator.py`

In [ ]:
# ============================================================
# Patch Generation — exact reproduction of patch_generator.py
# Uses the SAME extraction functions & process_row() logic,
# fed with the serialised fault_info strings from Step 5.
# ============================================================

# ---------- Error extraction (from patch_generator.py) ----------

def extract_ast_errors(ast_info_str: str) -> List[Tuple[int, int, str]]:
    """Extract error line numbers from ast_info JSON string."""
    if not ast_info_str or ast_info_str.strip() == '':
        return []
    try:
        info = json.loads(ast_info_str)
        if 'value' in info and info['value']:
            line_num = int(info['value'])
            error_type = info.get('type', 'AST Error')
            return [(line_num, line_num, f"ast: {error_type}")]
    except (json.JSONDecodeError, ValueError, TypeError) as e:
        print(f"Warning: Failed to parse ast_info: {ast_info_str[:100]}... Error: {e}")
    return []


def extract_cfg_errors(cfg_info_str: str) -> List[Tuple[int, int, str]]:
    """Extract error line ranges from cfg_info string (list-of-dicts via ast.literal_eval)."""
    if not cfg_info_str or cfg_info_str.strip() == '':
        return []
    try:
        info_list = ast.literal_eval(cfg_info_str)
        if not isinstance(info_list, list):
            return []
        errors = []
        for item in info_list:
            if isinstance(item, dict) and 'start_line' in item and 'end_line' in item:
                start_line = int(item['start_line'])
                end_line = int(item['end_line'])
                error_type = item.get('type', 'CFG Error')
                errors.append((start_line, end_line, f"cfg: {error_type}"))
        return errors
    except (SyntaxError, ValueError, TypeError) as e:
        print(f"Warning: Failed to parse cfg_info: {cfg_info_str[:100]}... Error: {e}")
    return []


def extract_lib_errors(lib_info_str: str) -> List[Tuple[int, int, str]]:
    """Extract error line numbers from lib_info string (list-of-dicts via ast.literal_eval)."""
    if not lib_info_str or lib_info_str.strip() == '':
        return []
    try:
        info_list = ast.literal_eval(lib_info_str)
        if not isinstance(info_list, list):
            return []
        errors = []
        for item in info_list:
            if isinstance(item, dict) and 'line' in item:
                line_num = int(item['line'])
                error_type = item.get('type', 'Library Error')
                errors.append((line_num, line_num, f"lib: {error_type}"))
        return errors
    except (SyntaxError, ValueError, TypeError) as e:
        print(f"Warning: Failed to parse lib_info: {lib_info_str[:100]}... Error: {e}")
    return []


def extract_dynamic_errors(dynamic_info_str: str) -> List[Tuple[int, int, str]]:
    """Extract error line numbers from dynamic_info JSON string."""
    if not dynamic_info_str or dynamic_info_str.strip() == '':
        return []
    try:
        info = json.loads(dynamic_info_str)
        if 'line_no' in info and info['line_no']:
            line_no_str = str(info['line_no']).strip()
            if line_no_str and line_no_str != '':
                # Convert to int (handle floats like "1.0")
                line_num = int(float(line_no_str))
                if line_num > 0:  # Valid line number
                    error_type = info.get('error_type', 'Dynamic Error')
                    return [(line_num, line_num, f"dynamic: {error_type}")]
    except (json.JSONDecodeError, ValueError, TypeError) as e:
        print(f"Warning: Failed to parse dynamic_info: {dynamic_info_str[:100]}... Error: {e}")
    return []


# ---------- Full-code patch generator (from patch_generator.py) ----------

def generate_full_patch(code: str, errors: List[Tuple[int, int, str]]) -> Optional[str]:
    """
    Generate the full code with error markers inserted at each error location.

    Args:
        code: The generated code
        errors: List of (start_line, end_line, error_type) tuples (1-indexed)

    Returns:
        Full code with error markers at each error location, or None if invalid
    """
    if not code:
        return None

    lines = code.split('\n')
    total_lines = len(lines)

    # Build lookup: line_idx -> markers before/after
    start_markers = {}  # idx -> list of error_type strings
    end_markers = {}    # idx -> list of error_type strings

    for start_line, end_line, error_type in errors:
        # Validate line numbers
        if start_line < 1 or end_line < 1 or start_line > total_lines or end_line > total_lines:
            print(f"Warning: Invalid line numbers {start_line}-{end_line} for code with {total_lines} lines")
            continue
        if start_line > end_line:
            print(f"Warning: start_line {start_line} > end_line {end_line}")
            continue

        start_markers.setdefault(start_line - 1, []).append(error_type)
        end_markers.setdefault(end_line - 1, []).append(error_type)

    # If no valid errors, return None
    if not start_markers:
        return None

    # Build the full patched code with markers
    patched_lines = []
    for i, line in enumerate(lines):
        if i in start_markers:
            for et in start_markers[i]:
                patched_lines.append(f"<<<< [ERROR START] ({et})")
        patched_lines.append(line)
        if i in end_markers:
            for et in end_markers[i]:
                patched_lines.append(f"[ERROR FINISH] ({et}) >>>>")

    return '\n'.join(patched_lines)


# ---------- process_row (from patch_generator.py) ----------

def process_row(ast_info_str, cfg_info_str, lib_info_str, dynamic_info_str, generated_code):
    """
    Process fault info strings + generated code → one combined patched code
    with all errors marked. Mirrors process_row() in patch_generator.py.

    Returns dict with patched_code, error_sources, error_types, error_lines
    or None if no errors found.
    """
    # Extract all errors from each source
    all_errors = []  # List of (source, start, end, error_type)

    ast_errors = extract_ast_errors(ast_info_str)
    for start, end, error_type in ast_errors:
        all_errors.append(('ast', start, end, error_type))

    # NOTE: CFG errors commented out for now — will be re-enabled later
    # cfg_errors = extract_cfg_errors(cfg_info_str)
    # for start, end, error_type in cfg_errors:
    #     all_errors.append(('cfg', start, end, error_type))

    lib_errors = extract_lib_errors(lib_info_str)
    for start, end, error_type in lib_errors:
        all_errors.append(('lib', start, end, error_type))

    dynamic_errors = extract_dynamic_errors(dynamic_info_str)
    for start, end, error_type in dynamic_errors:
        all_errors.append(('dynamic', start, end, error_type))

    # If no errors found, skip
    if not all_errors:
        return None

    # Build the list of (start, end, error_type) for generate_full_patch
    error_tuples = [(start, end, etype) for _, start, end, etype in all_errors]

    # Generate a single patched code with ALL error markers in the full code
    patched_code = generate_full_patch(generated_code, error_tuples)

    # Skip if patch generation failed
    if patched_code is None:
        return None

    # Aggregate error metadata
    error_sources = ','.join(source for source, _, _, _ in all_errors)
    error_types = ','.join(etype for _, _, _, etype in all_errors)
    error_lines = ','.join(f"{start}-{end}" for _, start, end, _ in all_errors)

    return {
        'patched_code': patched_code,
        'error_sources': error_sources,
        'error_types': error_types,
        'error_lines': error_lines,
    }


# ============================================================
# Run patch generator — fed with fault_info strings from Step 5
# ============================================================

patch_result = process_row(
    ast_info_str   = fault_info["ast_info"],
    cfg_info_str   = fault_info["cfg_info"],
    lib_info_str   = fault_info["lib_info"],
    dynamic_info_str = fault_info["dynamic_info"],
    generated_code = buggy_code,
)

patched_code = patch_result["patched_code"] if patch_result else None

# Keep a reference to the primary error line for downstream use
error_line = None
if dynamic_result["line_number"]:
    error_line = int(dynamic_result["line_number"])
elif ast_result["line"]:
    error_line = ast_result["line"]

print("=" * 60)
print("PATCH GENERATION RESULT")
print("=" * 60)
if patch_result:
    print(f"Error sources: {patch_result['error_sources']}")
    print(f"Error types:   {patch_result['error_types']}")
    print(f"Error lines:   {patch_result['error_lines']}")
    print()
    print("Full patched code (entire file with error markers):")
    print("-" * 40)
    for i, line in enumerate(patched_code.split('\n'), 1):
        print(f"  {i:>3} | {line}")
    print("-" * 40)
else:
    print(">> Could not generate patch (no valid errors identified).")

## Step 7 — KG Suggestions (DS-KG Knowledge Graph)

Load the **Knowledge Graph** (built from pandas, numpy, scipy, sklearn, etc.) and query it for API suggestions relevant to the detected error. The KG provides correct function signatures, parameter lists, and related methods.

Sources: `APR/DS-KG/UTIL/kg_util.py`, `APR/DS-KG/kg_context.py`

In [ ]:
# ============================================================
# DS-KG: Knowledge Graph Utilities — from kg_util.py
# ============================================================

# --- KG Loading ---
KG_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "Documents", "FYP-26", "APR", "DS-KG")

# Try multiple possible paths for the KG JSON files
possible_kg_dirs = [
    os.path.join("APR", "DS-KG"),                # relative to notebook
    os.path.join(os.getcwd(), "APR", "DS-KG"),    # absolute from cwd
    KG_DIR,                                        # from home
]

def load_kgs(kg_dir=None):
    """Load all kg_*.json files and merge into a single KG dict."""
    kg = {"functions": {}, "classes": {}}

    # Find the right directory
    search_dir = kg_dir
    if not search_dir:
        for d in possible_kg_dirs:
            if os.path.isdir(d) and glob.glob(os.path.join(d, "kg_*.json")):
                search_dir = d
                break

    if not search_dir:
        print("WARNING: Could not find KG JSON files!")
        return kg

    for filepath in glob.glob(os.path.join(search_dir, "kg_*.json")):
        with open(filepath, "r", encoding="utf8") as f:
            data = json.load(f)
            kg["functions"].update(data.get("functions", {}))
            kg["classes"].update(data.get("classes", {}))

    return kg


KG = load_kgs()
print(f"KG loaded: {len(KG['functions'])} functions, {len(KG['classes'])} classes")


# --- Error Detectors ---
def detect_name_error(msg):
    m = re.search(r"name '(.+?)' is not defined", msg)
    return m.group(1) if m else None

def detect_attribute_error(msg):
    m = re.search(r"'(.+?)' object has no attribute '(.+?)'", msg)
    return m.groups() if m else None

def detect_type_error(msg):
    m = re.search(r"(?:\w+\.)?(\w+)\(\) (?:got an unexpected keyword argument|takes?\b)", msg)
    if m:
        return m.group(1)
    m = re.search(r"(?:\w+\.)?(\w+)\(\) missing \d+ required", msg)
    if m:
        return m.group(1)
    return None

def detect_module_not_found(msg):
    """Stub for ModuleNotFoundError detection (not exercised for this AttributeError example)."""
    m = re.search(r"No module named '(.+?)'", msg)
    return m.group(1) if m else None


# --- Helpers ---
def rank(symbol, candidates):
    return get_close_matches(symbol, candidates, n=2, cutoff=0.85)

def build_function(name, node):
    return {
        "api": f"{node.get('module','')}.{name}",
        "type": node["node_type"],
        "required_params": node["parameters"]["required"],
        "optional_params": node["parameters"]["optional"],
        "description": node.get("description", "")
    }

def build_method(name, node):
    return {
        "api": f"{node['belongs_to']}.{name}",
        "type": "method",
        "belongs_to": node["belongs_to"],
        "required_params": node["parameters"]["required"],
        "optional_params": node["parameters"]["optional"],
        "description": node.get("description", "")
    }

def build_class(name, node):
    return {
        "api": name,
        "type": "class",
        "methods": node.get("methods", [])[:10],
        "attributes": node.get("attributes", [])[:10],
        "description": node.get("description", "")
    }


# --- Suggestion Engines ---
def suggest_name(symbol):
    out = []
    if symbol in KG["functions"]:
        node = KG["functions"][symbol]
        if node["node_type"] == "function":
            out.append(build_function(symbol, node))
        else:
            out.append(build_method(symbol, node))
    if symbol in KG["classes"]:
        out.append(build_class(symbol, KG["classes"][symbol]))
    return out[:2]

def suggest_attribute(cls, attr):
    out = []
    if cls in KG["classes"]:
        class_node = KG["classes"][cls]
        for m in rank(attr, class_node["methods"]):
            node = KG["functions"].get(m)
            if node:
                entry = build_method(m, node)
                entry["api"] = f"{cls}.{m}"
                entry["belongs_to"] = cls
                out.append(entry)
        for a in rank(attr, class_node["attributes"]):
            out.append({
                "api": f"{cls}.{a}",
                "type": "attribute",
                "belongs_to": cls
            })
    return out[:2]

def suggest_type(func):
    out = []
    if func in KG["functions"]:
        node = KG["functions"][func]
        if node["node_type"] == "function":
            out.append(build_function(func, node))
        else:
            out.append(build_method(func, node))
    return out[:2]

def suggest_module(mod):
    """Stub for module suggestions (not exercised for this example)."""
    return []


# ============================================================
# DS-KG: Context Provider — from kg_context.py
# ============================================================

# Regex to match patch_generator markers (with optional error-type annotation):
#   <<<< [ERROR START] (source: ErrorType)  ...code...  [ERROR FINISH] (source: ErrorType) >>>>
_MARKER_RE = re.compile(
    r"<<<<\s*\[ERROR START\]\s*(?:\([^)]*\))?\s*\n?(.*?)\n?\s*\[ERROR FINISH\]\s*(?:\([^)]*\))?\s*>>>>",
    re.DOTALL,
)

_ERROR_CATEGORIES = {"NameError", "AttributeError", "TypeError", "ModuleNotFoundError"}


def extract_error_region(code):
    """Return code between error markers, plus surrounding context.
    Handles multiple error regions by concatenating all of them."""
    matches = list(_MARKER_RE.finditer(code))
    if not matches:
        return {"region": "", "context_before": "", "context_after": ""}

    # Collect all marked error regions
    regions = [m.group(1).strip() for m in matches]
    region = "\n".join(regions)

    # Context from first match (before) and last match (after)
    before_text = code[:matches[0].start()]
    after_text = code[matches[-1].end():]

    before_lines = before_text.rstrip("\n").split("\n")
    after_lines = after_text.lstrip("\n").split("\n")

    context_before = "\n".join(before_lines[-3:]).strip()
    context_after = "\n".join(after_lines[:3]).strip()

    return {"region": region, "context_before": context_before, "context_after": context_after}


def classify_error(error_info):
    """Determine canonical error category from error_info dict."""
    etype = str(error_info.get("error_type", "")).strip()
    for cat in _ERROR_CATEGORIES:
        if cat in etype:
            return cat

    msg = str(error_info.get("error_message", ""))
    if detect_name_error(msg): return "NameError"
    if detect_attribute_error(msg): return "AttributeError"
    if detect_module_not_found(msg): return "ModuleNotFoundError"
    if detect_type_error(msg): return "TypeError"

    status = str(error_info.get("status", ""))
    if status:
        if detect_name_error(status): return "NameError"
        if detect_attribute_error(status): return "AttributeError"
        if detect_module_not_found(status): return "ModuleNotFoundError"
        if detect_type_error(status): return "TypeError"

    return "Unknown"


class _SymbolExtractor(ast.NodeVisitor):
    """Walk AST to collect imports, attribute accesses, and calls."""
    def __init__(self):
        self.imports = []
        self.attributes = []
        self.calls = []

    def visit_Import(self, node):
        for alias in node.names:
            self.imports.append((alias.name, alias.asname or alias.name))
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        mod = node.module or ""
        for alias in node.names:
            self.imports.append((f"{mod}.{alias.name}", alias.asname or alias.name))
        self.generic_visit(node)

    def visit_Attribute(self, node):
        obj_name = _resolve_name(node.value)
        if obj_name:
            self.attributes.append((obj_name, node.attr))
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            self.calls.append(node.func.id)
        elif isinstance(node.func, ast.Attribute):
            self.calls.append(node.func.attr)
        self.generic_visit(node)


def _resolve_name(node):
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        parent = _resolve_name(node.value)
        if parent:
            return f"{parent}.{node.attr}"
    return None


def analyze_error_region(error_region, full_code=""):
    result = {"imports": [], "attributes": [], "calls": []}
    code_to_parse = full_code if full_code else error_region
    try:
        tree = ast.parse(code_to_parse)
    except SyntaxError:
        try:
            tree = ast.parse(error_region)
        except SyntaxError:
            return result

    extractor = _SymbolExtractor()
    extractor.visit(tree)
    result["imports"] = extractor.imports
    result["attributes"] = extractor.attributes
    result["calls"] = extractor.calls
    return result


def _resolve_class_from_imports(obj_name, imports):
    if obj_name in KG["classes"]:
        return obj_name
    return None


def query_kg(error_type, error_info, parsed_symbols):
    """Query the KG for suggestions based on error type."""
    suggestions = []
    msg = str(error_info.get("error_message", ""))
    status = str(error_info.get("status", ""))
    combined_msg = msg or status

    libapi_details = error_info.get("libapi_details", [])
    if isinstance(libapi_details, str):
        try:
            libapi_details = json.loads(libapi_details)
        except (json.JSONDecodeError, TypeError):
            libapi_details = []

    if error_type == "NameError":
        symbol = detect_name_error(combined_msg)
        if symbol:
            suggestions.extend(suggest_name(symbol))
        for call_name in parsed_symbols.get("calls", []):
            if call_name not in KG["functions"] and call_name not in KG["classes"]:
                suggestions.extend(suggest_name(call_name))
        for detail in libapi_details:
            if detail.get("type") == "name_error":
                name = detail.get("name", "")
                if name and name != symbol:
                    suggestions.extend(suggest_name(name))

    elif error_type == "AttributeError":
        parsed = detect_attribute_error(combined_msg)
        if parsed:
            cls, attr = parsed
            suggestions.extend(suggest_attribute(cls, attr))
        for obj_name, attr_name in parsed_symbols.get("attributes", []):
            resolved = _resolve_class_from_imports(obj_name, parsed_symbols.get("imports", []))
            if resolved:
                suggestions.extend(suggest_attribute(resolved, attr_name))
        for detail in libapi_details:
            if detail.get("type") == "attribute_error":
                obj = detail.get("object", "")
                attr = detail.get("attribute", "")
                if obj and attr:
                    suggestions.extend(suggest_attribute(obj, attr))

    elif error_type == "TypeError":
        func = detect_type_error(combined_msg)
        if func:
            suggestions.extend(suggest_type(func))
        for call_name in parsed_symbols.get("calls", []):
            if call_name in KG["functions"]:
                suggestions.extend(suggest_type(call_name))
        for detail in libapi_details:
            if detail.get("type") == "type_error":
                fn = detail.get("function", "")
                if fn:
                    suggestions.extend(suggest_type(fn))

    elif error_type == "ModuleNotFoundError":
        mod = detect_module_not_found(combined_msg)
        if mod:
            suggestions.extend(suggest_module(mod))

    # Deduplicate
    seen = set()
    unique = []
    for s in suggestions:
        key = s.get("api", "")
        if key and key not in seen:
            seen.add(key)
            unique.append(s)
    return unique


def _format_suggestion(i, s):
    lines = [f"{i}. {s['api']} ({s['type']})"]
    if s.get("required_params"):
        lines.append(f"   - Required params: {', '.join(s['required_params'])}")
    if s.get("optional_params"):
        lines.append(f"   - Optional params: {', '.join(s['optional_params'])}")
    if s.get("belongs_to"):
        lines.append(f"   - Belongs to: {s['belongs_to']}")
    if s.get("methods"):
        lines.append(f"   - Methods: {', '.join(s['methods'][:5])}")
    if s.get("attributes"):
        lines.append(f"   - Attributes: {', '.join(s['attributes'][:5])}")
    if s.get("description"):
        lines.append(f"   - Description: {s['description']}")
    return "\n".join(lines)


def format_context(error_region_info, error_type, suggestions):
    """Build human-readable context summary for LLM prompt."""
    parts = []
    parts.append(f"Error Type: {error_type}")
    parts.append("")

    if error_region_info.get("region"):
        parts.append("Error Region:")
        for line in error_region_info["region"].split("\n"):
            parts.append(f"  {line}")
        parts.append("")

    if error_region_info.get("context_before"):
        parts.append("Context Before Error:")
        for line in error_region_info["context_before"].split("\n"):
            parts.append(f"  {line}")
        parts.append("")

    if suggestions:
        parts.append("KG API Suggestions:")
        for i, s in enumerate(suggestions, 1):
            parts.append(_format_suggestion(i, s))
            parts.append("")
    else:
        parts.append("No KG suggestions found for this error.")
        parts.append("")

    return "\n".join(parts).strip()


def get_repair_context(code, error_info):
    """Main API: extract error region, classify, query KG, format context."""
    region_info = extract_error_region(code)
    error_type = classify_error(error_info)

    clean_code = _MARKER_RE.sub(lambda m: m.group(1), code)
    parsed_symbols = analyze_error_region(region_info["region"], clean_code)

    suggestions = query_kg(error_type, error_info, parsed_symbols)
    context_summary = format_context(region_info, error_type, suggestions)

    return {
        "error_region": region_info["region"],
        "error_type": error_type,
        "suggestions": suggestions,
        "context_summary": context_summary,
    }


# ============================================================
# Run KG suggestion pipeline on the patched code
# ============================================================

error_info_for_kg = {
    "error_type": dynamic_result["error_type"],
    "error_message": dynamic_result["error_message"],
    "status": fault_info["status"],
    "libapi_details": libapi_result["libapi_details"],
}

kg_context = get_repair_context(patched_code, error_info_for_kg)

print("=" * 60)
print("KG SUGGESTION RESULT")
print("=" * 60)
print(f"Error Type (classified): {kg_context['error_type']}")
print(f"Error Region: {kg_context['error_region']}")
print(f"Number of suggestions: {len(kg_context['suggestions'])}")
print()
if kg_context["suggestions"]:
    print("Suggestions:")
    for i, s in enumerate(kg_context["suggestions"], 1):
        print(f"  {_format_suggestion(i, s)}")
        print()
else:
    print("No KG suggestions found.")
print()
print("-" * 60)
print("FULL CONTEXT SUMMARY (for LLM prompt):")
print("-" * 60)
print(kg_context["context_summary"])

## Step 8 — Load Qwen2.5-Coder-3B-Instruct Model

Load the **Qwen2.5-Coder-3B-Instruct** model for code repair generation. This model specializes in code understanding and generation tasks.

In [ ]:
# ============================================================
# Load Qwen2.5-Coder-3B-Instruct
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

print(f"Loading model: {model_id}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

# Determine device
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)

if device != "cuda":
    model = model.to(device)

print(f"Model loaded successfully on {device}.")

## Step 9 — Construct Prompt & Generate Fix

Build a structured repair prompt that includes:
- The **full generated code with error markers** (entire file, error regions annotated inline)
- All error information from each analysis stage
- KG API suggestions

Then feed it to the Qwen model to generate the corrected code.

In [ ]:
# ============================================================
# Construct repair prompt and generate fix
# ============================================================
# The prompt gives the model the FULL original code plus the
# error-location markers from the patch generator. This way the
# model sees the complete program AND knows exactly where the
# error is. If the model still returns only a snippet,
# reconstruct_full_code() merges it back into the original.

buggy_lines = buggy_code.strip().split('\n')
num_buggy_lines = len(buggy_lines)

# Build error summary
error_summary_parts = []
if ast_result["error_type"]:
    error_summary_parts.append(f"AST: {ast_result['error_type']} at line {ast_result['line']}")
if cfg_result["cfg_details"]:
    for d in cfg_result["cfg_details"]:
        error_summary_parts.append(f"CFG: {d['type']}")
if libapi_result["libapi_details"]:
    for d in libapi_result["libapi_details"]:
        error_summary_parts.append(f"LibAPI: {d['type']}")
if dynamic_result["status"] == "failed":
    error_summary_parts.append(
        f"Dynamic: {dynamic_result['error_type']} at line {dynamic_result['line_number']}: "
        f"{dynamic_result['error_message']}"
    )
error_summary = "; ".join(error_summary_parts) if error_summary_parts else "No static errors"

kg_summary = kg_context['context_summary']


def clean_model_output(raw):
    """Strip markdown fences and trailing non-code text."""
    code = raw.strip()
    if code.startswith("```python"):
        code = code[len("```python"):]
    if code.startswith("```"):
        code = code[3:]
    # Remove everything after a closing fence
    if "```" in code:
        code = code[:code.index("```")]
    code = code.strip()
    # Remove trailing blank lines
    lines = code.split('\n')
    while lines and not lines[-1].strip():
        lines.pop()
    return '\n'.join(lines)


def is_complete(code_str):
    """Check if the output looks like the complete fixed program."""
    lines = code_str.strip().split('\n')
    if len(lines) < num_buggy_lines:
        return False
    if lines[0].strip() != buggy_lines[0].strip():
        return False
    return True


def reconstruct_full_code(buggy_code, model_output, error_start, error_end):
    """
    Ensure the output is the complete fixed program.
    If the model returned only a snippet, merge it back into the
    original buggy code by replacing the error region.
    """
    buggy_lines_local = buggy_code.strip().split('\n')
    output_lines = model_output.strip().split('\n')

    # If output already looks like the complete program, return as-is
    if is_complete(model_output):
        return model_output.strip()

    # Model returned a snippet — strip context lines it echoed back
    fix_lines = output_lines[:]

    # Strip leading context lines (lines before the error that the model echoed)
    idx = error_start - 2  # 0-indexed line just before error
    while idx >= 0 and fix_lines:
        if fix_lines[0].strip() == buggy_lines_local[idx].strip():
            fix_lines.pop(0)
            idx -= 1
        else:
            break

    # Strip trailing context lines (lines after the error that the model echoed)
    idx = error_end  # 0-indexed line just after error
    while idx < len(buggy_lines_local) and fix_lines:
        if fix_lines[-1].strip() == buggy_lines_local[idx].strip():
            fix_lines.pop()
            idx += 1
        else:
            break

    # Reconstruct: unchanged prefix + fixed lines + unchanged suffix
    full = buggy_lines_local[:error_start - 1] + fix_lines + buggy_lines_local[error_end:]
    return '\n'.join(full)


def generate_with_messages(msgs, prefix_lines=None, temp=0.2):
    """Run generation. Optionally prefix-prime with given lines."""
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    prefix_str = ""
    if prefix_lines:
        prefix_str = "\n".join(prefix_lines) + "\n"
        text += prefix_str

    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs, max_new_tokens=512,
            temperature=temp, top_p=0.95, do_sample=True,
        )
    new_ids = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    gen_text = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0]
    return clean_model_output(prefix_str + gen_text)


# ============================================================
# Build the repair prompt
# ============================================================
# The prompt gives the model a SINGLE view: the complete program
# with inline <<<< [ERROR START] / [ERROR FINISH] >>>> markers
# at every detected fault location (matching patched_code.csv).
# Plus error details and KG API suggestions for context.

system_prompt = (
    "You are a code repair assistant. You fix buggy Python code. "
    "Always return the COMPLETE fixed program — every line from the "
    "first import to the last statement. Output ONLY Python code, "
    "no explanations, no markdown fences."
)

user_prompt = f"""Fix the buggy Python code below.

## Full Generated Code with Error Markers:
The code below is the COMPLETE program. Lines between <<<< [ERROR START] and [ERROR FINISH] >>>> markers are the faulty regions that need fixing.

{patched_code}

## Error Details:
{error_summary}

## KG API Suggestions:
{kg_summary}

## Instructions:
- Fix ONLY the errors in the marked regions
- Return the COMPLETE fixed program (ALL lines, starting from the imports)
- Remove the error markers — return clean Python code only
- Do NOT return just the fixed snippet — return the entire program with all fixes applied
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# --- Display the prompt ---
print("=" * 60)
print("REPAIR PROMPT")
print("=" * 60)
print(user_prompt)

# --- Generate the fix ---
print("=" * 60)
print("GENERATING FIX...")
print("=" * 60)

result_code = generate_with_messages(messages, prefix_lines=buggy_lines[:1], temp=0.2)

print(f"\nModel raw output ({len(result_code.strip().split(chr(10)))} lines):")
print("-" * 40)
for ln in result_code.strip().split('\n'):
    print(f"  | {ln}")
print("-" * 40)

# --- Ensure we have the complete program ---
if is_complete(result_code):
    full_fixed_code = result_code.strip()
    print("\n>> Model returned the COMPLETE program directly.")
else:
    # Model returned a snippet — reconstruct by merging into original code
    full_fixed_code = reconstruct_full_code(
        buggy_code, result_code, error_line, error_line
    )
    print(f"\n>> Model returned a snippet — reconstructed into full program.")

print("\n" + "=" * 60)
print("FIXED CODE (generated by Qwen Coder)")
print("=" * 60)
for i, line in enumerate(full_fixed_code.strip().split('\n'), 1):
    print(f"  {i:>3} | {line}")
print("=" * 60)

## Step 10 — Final Comparison

Side-by-side comparison of the **original buggy code** vs. the **model-generated fix**, along with a summary of the full pipeline.

In [ ]:
# ============================================================
# Final Comparison: Buggy vs Fixed Code
# ============================================================

print("=" * 70)
print("FINAL COMPARISON")
print("=" * 70)

# --- Buggy Code ---
print("\n  ORIGINAL BUGGY CODE:")
print("  " + "-" * 50)
for i, line in enumerate(buggy_code.strip().split('\n'), 1):
    marker = " >> " if str(i) == dynamic_result.get("line_number", "") else "    "
    print(f"  {marker}{i:>3} | {line}")
print("  " + "-" * 50)

# --- Fixed Code ---
print("\n  FULL FIXED CODE:")
print("  " + "-" * 50)
buggy_set = {l.strip() for l in buggy_lines}
for i, line in enumerate(full_fixed_code.strip().split('\n'), 1):
    is_new = line.strip() not in buggy_set
    marker = " ** " if is_new else "    "
    print(f"  {marker}{i:>3} | {line}")
print("  " + "-" * 50)
print("  (** = changed line)")

# --- Verify the fix ---
print("\n" + "=" * 70)
print("VERIFICATION: Re-running fixed code...")
print("=" * 70)

# Original error detected by the pipeline
original_error_type = dynamic_result.get("error_type", "")
original_error_msg = dynamic_result.get("error_message", "")

try:
    verify_env = {"test_input": test_input}
    exec(full_fixed_code, verify_env)
    print(">> PASS: Fixed code executed without errors!")
    if "result" in verify_env:
        print(f">> Result type: {type(verify_env['result']).__name__}")
        print(f">> Result preview:\n{verify_env['result']}")
except Exception as e:
    new_error_type = type(e).__name__
    new_error_msg = str(e)
    # Check if the SAME error persists (fix failed) or a DIFFERENT one surfaced
    if new_error_type == original_error_type and original_error_msg in new_error_msg:
        print(f">> FAIL: Original error persists — {new_error_type}: {new_error_msg}")
        print("   The model's fix did not resolve the detected hallucination.")
    else:
        print(f">> PARTIAL FIX: Original {original_error_type} resolved!")
        print(f"   Secondary error exposed: {new_error_type}: {new_error_msg}")
        print("   (A pre-existing issue in the code, previously masked by the primary error.)")

# --- Pipeline Summary ---
print("\n" + "=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
print(f"  Task:             DS0042 (DS1000 dataset)")
print(f"  Status:           Hallucinated")
print(f"  AST Analysis:     {'PASS' if ast_result['ast_parsed'] else 'FAIL'} — {ast_result['structural_error']} structural issues")
print(f"  CFG Analysis:     {cfg_result['unreachable_code']} unreachable, {cfg_result['missing_return']} missing returns")
print(f"  Library/API:      {libapi_result['total_libapi_errors']} errors detected")
print(f"  Dynamic:          {dynamic_result['error_type']} at line {dynamic_result['line_number']}")
print(f"  KG Suggestions:   {len(kg_context['suggestions'])} API suggestions provided")
print(f"  Error Type:       {kg_context['error_type']}")
print(f"  Model:            {model_id}")
print("=" * 70)